# Quick Start Guide

Lightning Rod's **Foresight** models return calibrated probability forecasts for any
forward-looking question through an OpenAI-compatible API.

This notebook shows the two ways to call it:
1. **Any OpenAI client**, pointed at Lightning Rod's `base_url`.
2. **The Python SDK helper** `lr.predict()`, which parses the structured answer for you.

Full reference: [docs.lightningrod.ai](https://docs.lightningrod.ai).

## Install

In [ ]:
%pip install lightningrod-ai python-dotenv

from IPython.display import clear_output
clear_output()

## Set up your API key

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [ ]:
from dotenv import load_dotenv

from lightningrod.utils import config



load_dotenv()

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

## Path 1: OpenAI-compatible client

Any OpenAI client works out of the box — just point `base_url` at Lightning Rod.

In [ ]:
from openai import OpenAI



client = OpenAI(

    api_key=api_key,

    base_url="https://api.lightningrod.ai/v1/openai",

)



response = client.chat.completions.create(

    model="foresight-v4",

    messages=[

        {"role": "user", "content": "Will the Fed cut rates at its next meeting?"},

    ],

    extra_body={"research": True},  # Auto-research the most relevant prediction context

)



print(response.choices[0].message.content)

Lightning Rod adds three optional extensions on top of the standard OpenAI fields, passed via `extra_body`:

- `answer_type` — request a machine-readable answer (`<answer>...</answer>`) in a specific shape (`binary`, `continuous`, `multiple_choice`, `free_response`)
- `research` — `True` to query all default sources, or a list like `["perplexity", "google_news"]` to restrict providers
- `reasoning_effort` — `"low"`, `"medium"` (default) or `"high"`

See the [OpenAI API guide](https://docs.lightningrod.ai/forecasting/openai) for response shapes per `answer_type`.

In [ ]:
response = client.chat.completions.create(

    model="foresight-v4",

    messages=[

        {"role": "user", "content": "Will the Fed cut rates by 25bp in March 2026?"},

    ],

    extra_body={

        "answer_type": "binary",

        "research": {"sources": ["perplexity", "google_news"]},

        "reasoning_effort": "low",

    },

)



print(response.choices[0].message.content)  # ... <answer>0.62</answer>

## Path 2: SDK helper — `lr.predict()`

`lr.predict()` wraps the same API and parses the structured answer into typed fields for you.

In [ ]:
import lightningrod as lr



client = lr.LightningRod(api_key=api_key)



result = client.predict(

    "Will the Fed cut rates by 25bp in March 2026?",

    answer_type="binary",

    research=["perplexity", "google_news"],

    reasoning_effort="low",

)



print(result.binary.probability)  # e.g. 0.62

print(result.content)             # full response, including <answer> tags

print(result.sources)             # citations gathered during research

print(result.usage)               # token counts and cost fields

## Next steps

- [Quickstart](https://docs.lightningrod.ai/forecasting/quickstart) — available models and prediction context
- [OpenAI API](https://docs.lightningrod.ai/forecasting/openai) — full reference for the raw OpenAI-compatible client
- [Python SDK](https://docs.lightningrod.ai/forecasting/sdk) — full reference for `lr.predict()` and `PredictionResult`
- [Recipes](https://docs.lightningrod.ai/forecasting/recipes) — writing good forecasting prompts
- [Enterprise Platform](https://docs.lightningrod.ai/platform/overview) — generate datasets and fine-tune your own forecasting models